# TP5 : Diagnostic par les traces

**Contexte :** assistant du service client de Lagune Banque.

Vous allez reproduire trois erreurs d'agent, en retrouver la cause dans les traces, appliquer une parade et vérifier que le comportement est corrigé.

| Erreur | Ce qu'on cherche dans la trace | Parade |
|---|---|---|
| 1. Une conversation qui ne se termine pas | le tool rappelé sans progresser | limite d'appels |
| 2. Un mauvais tool | le tool appelé et celui qui était attendu | descriptions corrigées |
| 3. Une date que personne n'a donnée | l'origine de l'argument | validation dans le tool et question au client |

Complétez les `# TODO`. Références : [agents](https://docs.langchain.com/oss/python/langchain/agents), [traces Langfuse](https://langfuse.com/docs/integrations/langchain/tracing).

## <a id="configuration"></a>Configuration

In [ ]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path("."), Path(".."), Path("../..")] if (p / "code_partage").exists()).resolve()
sys.path[:0] = [str(RACINE / "code_partage"), str(RACINE / "03_agents" / "socle")]

import json
import warnings
from datetime import datetime
from typing import Optional

import pandas as pd
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.errors import GraphRecursionError

import tools as T
from base import AUJOURDHUI, HIER, KOUASSI, Context, new_model, reset_db, sql
from config import prepare
from observability import clear_langfuse, langfuse_callbacks, langfuse_enabled
from tracing import Tracer

warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

In [ ]:
prepare()
print("Langfuse :", "configuré, les traces sont aussi envoyées" if langfuse_enabled() else "non configuré : les traces restent locales")

## <a id="syntaxe"></a>Syntaxe

Ce que le TP utilise pour lire et contraindre un agent :

- **Résultat de `agent.invoke(...)`** : un dict dont `result["messages"]` est la liste ordonnée des messages. `HumanMessage` : le client. `AIMessage` : le modèle, avec `.content` (la réponse) et `.tool_calls`, une liste de dicts `{"name", "args", "id"}` pour les tools demandés. `ToolMessage` : le résultat d'un tool.
- **Limite d'appels** : `ToolCallLimitMiddleware(tool_name=..., run_limit=..., exit_behavior="end")` plafonne les appels d'un tool par conversation et termine proprement.
- **Tool qui lit l'état** : dans un tool, `runtime.state["messages"]` donne les messages de la conversation, `runtime.context` le contexte de l'application.

```python
for m in result["messages"]:
    if isinstance(m, AIMessage) and m.tool_calls:
        print(m.tool_calls)   # [{"name": "get_schedule", "args": {}, "id": "call_1", "type": "tool_call"}]
```

## <a id="lecture-des-traces"></a>Lecture des traces

`Tracer` collecte en local chaque étape : la décision du modèle (tool demandé ou réponse), l'appel du tool avec ses arguments et son résultat, la durée et les tokens. Si Langfuse est configuré, la même exécution y est aussi envoyée.

L'agent est celui du TP1 : mêmes tools, même system prompt.

Le system prompt du TP1 (`T.SYSTEM_PROMPT`, dans `socle/tools.py`) fixe le rôle de l'agent, ses règles et la procédure pas à pas. L'erreur 2 le reprend sans cette procédure.

In [ ]:
print(T.SYSTEM_PROMPT)

In [ ]:
def run(agent, message, client_id="CLI-0002", limit=25):
    reset_db()
    tracer = Tracer()
    callbacks, meta = langfuse_callbacks(session_id="tp5", utilisateur=client_id, tags=["tp5"], nom="m3-tp5-traces")
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": message}]},
                              {"callbacks": [tracer, *callbacks], "metadata": meta, "recursion_limit": limit}, context=Context(client_id=client_id))
        return tracer, result, None
    except GraphRecursionError:
        return tracer, None, "GraphRecursionError"


def final_answer(result):
    return str(result["messages"][-1].content) if result else None


def build_agent(check=None, descriptions=None, limiter=None, prompt=T.SYSTEM_PROMPT):
    descriptions = descriptions or {}
    tools = []
    for t in T.TOOLS:
        if t.name == "check_lm_transaction":
            t = (check or t)
        tools.append(t.model_copy(update={"description": descriptions.get(t.name, t.description)}))
    middleware = [limiter] if limiter else []
    return create_agent(new_model(), tools, system_prompt=prompt, context_schema=Context, middleware=middleware)


TRACE_OK, RESULT_OK, _ = run(build_agent(), KOUASSI)
TRACE_OK.table()

Chaque ligne `model` est une décision (le tool demandé ou la réponse finale) ; la ligne `tool` qui suit est l'appel exécuté. C'est là qu'on cherche : quel tool, avec quels arguments, pour quel résultat.

**TODO 1** : `RESULT_OK` est le résultat brut de `agent.invoke`. La cellule parcourt `RESULT_OK["messages"]` et construit `TOOLS_DEMANDES`, la liste des `(nom, arguments)` de chaque tool demandé par le modèle, dans l'ordre. Complétez la ligne qui lit, dans un `AIMessage` (`m`), la liste des tools demandés (voir la section Syntaxe).

In [ ]:
TOOLS_DEMANDES = []
for m in RESULT_OK["messages"]:
    if isinstance(m, AIMessage):
        appels = []   # TODO 1 : remplacez [] par la liste des tools demandés dans ce message
        for appel in appels:
            TOOLS_DEMANDES.append((appel["name"], appel["args"]))
print(TOOLS_DEMANDES)

## <a id="erreur-1-boucle-dappels"></a>Erreur 1 : boucle d'appels

Le tool `check_lm_transaction` interroge une API de paiement. Ici elle répond toujours « recherche en cours, rappelez ce tool » et ne donne jamais de résultat. Rien dans l'agent ne dit quand arrêter : seul le plafond du framework (`recursion_limit`, 30 étapes ici) interrompt la conversation.

In [ ]:
@tool("check_lm_transaction", description=T.check_lm_transaction.description)
def check_pending(date: str, runtime: ToolRuntime[Context], montant: Optional[int] = None) -> str:
    return json.dumps({"statut": "en_cours", "message": "Recherche en cours de traitement : rappelez ce tool pour obtenir le résultat."}, ensure_ascii=False)


TRACE_LOOP, RESULT_LOOP, ERREUR_LOOP = run(build_agent(check=check_pending), KOUASSI, limit=30)
print("Arrêt :", ERREUR_LOOP or "réponse finale")
TRACE_LOOP.table().tail(8)

**TODO 2** : la cellule compte les appels de chaque tool dans `TRACE_LOOP`. Indiquez dans `TOOL_BOUCLE` le nom du tool qui tourne en boucle ; `APPELS_BOUCLE`, son nombre d'appels, en est déduit.

In [ ]:
NOMS = [s["name"] for s in TRACE_LOOP.tool_calls()]
print(pd.Series(NOMS, dtype=object).value_counts().to_string())
TOOL_BOUCLE = None   # TODO 2 : remplacez None par le nom du tool, entre guillemets
APPELS_BOUCLE = NOMS.count(TOOL_BOUCLE)
print(TOOL_BOUCLE, APPELS_BOUCLE)

**TODO 3** : la parade est une limite d'appels par conversation. Créez `LIMITER`, un `ToolCallLimitMiddleware` sur le tool `check_lm_transaction`, avec `run_limit` entre 1 et 5 et `exit_behavior="end"` : au-delà, le tool n'est plus exécuté et la conversation se termine avec un message qui explique pourquoi. Dans un vrai service, cette sortie déclenche un transfert à un conseiller.

In [ ]:
LIMITER = None   # TODO 3
LIMITE_APPELS = LIMITER.run_limit if LIMITER else None
TRACE_LIMIT, RESULT_LIMIT, ERREUR_LIMIT = run(build_agent(check=check_pending, limiter=LIMITER), KOUASSI, limit=30)
APPELS_LIMIT = [s["name"] for s in TRACE_LIMIT.tool_calls()].count("check_lm_transaction")
print(f"Sans limite : {APPELS_BOUCLE} appels, arrêt par {ERREUR_LOOP or 'réponse finale'}")
print(f"Avec limite : {APPELS_LIMIT} appels de check_lm_transaction, arrêt par {ERREUR_LIMIT or 'réponse finale'}")
print(final_answer(RESULT_LIMIT))

## <a id="erreur-2-mauvais-outil"></a>Erreur 2 : mauvais tool

Une description copiée-collée au mauvais endroit : `check_lm_transaction` porte la description de `check_platform_incident`, et inversement. Les noms sont justes, les descriptions non. Le system prompt est celui du TP1 **sans la procédure pas à pas** : sans elle, ce sont les descriptions qui guident le choix du tool.

On pose quatre formulations de la même demande. Le tool attendu en premier est `check_lm_transaction`.

In [ ]:
DESCRIPTIONS_ORIGINALES = {t.name: t.description for t in T.TOOLS}
INTERVERTIES = {"check_lm_transaction": DESCRIPTIONS_ORIGINALES["check_platform_incident"],
                "check_platform_incident": DESCRIPTIONS_ORIGINALES["check_lm_transaction"]}
SANS_PROCEDURE = T.SYSTEM_PROMPT.split("Cas particulier")[0].strip()
FORMULATIONS = ["Mon paiement de 31 500 FCFA du 15 novembre est-il bien arrivé ?", "Est-ce que mon remboursement d'hier a été reçu ?",
                "J'ai payé 31 500 FCFA le 15 novembre, a-t-il été reçu ?", "Je voudrais savoir si mon virement du 15 novembre est passé."]

AGENT_INTERVERTI = build_agent(descriptions=INTERVERTIES, prompt=SANS_PROCEDURE)
ESSAIS = [run(AGENT_INTERVERTI, q)[0] for q in FORMULATIONS]
TRACE_BAD = next((t for t in ESSAIS if t.decisions()[0] != ["check_lm_transaction"]), ESSAIS[0])
print("Première décision du modèle, par formulation :", [t.decisions()[0] for t in ESSAIS])
TRACE_BAD.table().head(6)

La trace affichée est la première dont la décision initiale n'est pas exactement `check_lm_transaction`. Comparez le tool appelé, son résultat et la description qu'il porte.

**TODO 4** : dans `TRACE_BAD`, relevez la première décision du modèle (`PREMIERE_DECISION`, une liste de noms de tools : `TRACE_BAD.decisions()` les donne) et dites si c'était le bon choix (`BON_CHOIX`, un booléen : la liste est exactement `["check_lm_transaction"]`).

In [ ]:
PREMIERE_DECISION, BON_CHOIX = None, None
print(PREMIERE_DECISION, BON_CHOIX)

**TODO 5** : écrivez les deux descriptions correctes, qui ne se recouvrent pas. Dites à quoi sert chaque tool, le format de ses paramètres, quand l'appeler et ce qu'il ne fait pas.

In [ ]:
DESC_TRANSACTION = "TODO 5"
DESC_INCIDENT = "TODO 5"
CORRIGEES = {"check_lm_transaction": DESC_TRANSACTION, "check_platform_incident": DESC_INCIDENT}

Mesure : les quatre formulations, `RUNS` exécutions chacune. On compte les conversations dont la première décision est exactement `check_lm_transaction`.

In [ ]:
RUNS = 2


def rate_right_first(agent):
    ok = [run(agent, q)[0].decisions()[0] == ["check_lm_transaction"] for q in FORMULATIONS for _ in range(RUNS)]
    return sum(ok), len(ok)


AVANT_2 = rate_right_first(AGENT_INTERVERTI)
APRES_2 = rate_right_first(build_agent(descriptions=CORRIGEES, prompt=SANS_PROCEDURE))
print(f"Descriptions interverties : bon tool en premier dans {AVANT_2[0]} conversations sur {AVANT_2[1]}")
print(f"Descriptions corrigées    : bon tool en premier dans {APRES_2[0]} conversations sur {APRES_2[1]}")

## <a id="erreur-3-une-date-que-personne-na-donnee"></a>Erreur 3 : une date que personne n'a donnée

Le client ne donne pas la date de son remboursement. Or `check_lm_transaction` exige une date.

In [ ]:
NODATE = "J'ai remboursé 31 500 FCFA via Lagune Money mais mon dossier indique toujours un impayé."
TRACE_DATE, RESULT_DATE, _ = run(build_agent(), NODATE)
TRACE_DATE.table()

`argument_origin(value, message, previous_results)` est fournie. Elle renvoie `"client"` si `value` figure dans le message du client, `"tool"` s'il figure dans un résultat de tool précédent, `"inventée"` sinon. On l'applique à la date du premier appel de `check_lm_transaction`. Une date reprise d'un résultat de tool (la date du jour, par exemple) n'est pas non plus la date du remboursement.

In [ ]:
def argument_origin(value, message, previous_results):
    if value in message:
        return "client"
    if any(value in r for r in previous_results):
        return "tool"
    return "inventée"


APPELS_DATE = [s for s in TRACE_DATE.tool_calls() if s["name"] == "check_lm_transaction"]
DATE_APPELEE = APPELS_DATE[0]["args"].get("date") if APPELS_DATE else None
RESULTATS_AVANT = [s["result"] for s in TRACE_DATE.tool_calls()[:TRACE_DATE.tool_calls().index(APPELS_DATE[0])]] if APPELS_DATE else []
ORIGINE_DATE = argument_origin(DATE_APPELEE, NODATE, RESULTATS_AVANT) if DATE_APPELEE else None
print(f"date transmise au tool : {DATE_APPELEE} ; origine : {ORIGINE_DATE}")

`date_given(date, texts)` est fournie : vraie si `date` (AAAA-MM-JJ) se déduit des messages du client, c'est-à-dire si elle y figure telle quelle, si le client a dit « hier » ou « aujourd'hui » et que la date correspond, ou s'il a écrit le jour et le mois (« 15 novembre »).

In [ ]:
MOIS = {1: "janvier", 2: "février", 3: "mars", 4: "avril", 5: "mai", 6: "juin", 7: "juillet", 8: "août", 9: "septembre", 10: "octobre",
        11: "novembre", 12: "décembre"}


def date_given(date, texts):
    texte = " ".join(texts).lower()
    jour = datetime.fromisoformat(date)
    if date in texte:
        return True
    if "hier" in texte and date == HIER:
        return True
    if "aujourd'hui" in texte and date == AUJOURDHUI:
        return True
    return f"{jour.day} {MOIS[jour.month]}" in texte

**TODO 6** : la parade est une validation **dans le tool**, pas dans le prompt : le tool refuse une date que le client n'a pas donnée, avec un message qui dit quoi faire. `check_validated` remplace `check_lm_transaction` : il récupère les messages du client (`messages_client`, les `HumanMessage` de `runtime.state["messages"]`), refuse la date si elle n'y figure pas, et sinon lit les transactions comme avant. Complétez la ligne `date_valide` avec la vérification `date_given(date, messages_client)`. Quand la date est refusée, l'`indication` renvoyée au modèle lui dit de redemander la date au client.

In [ ]:
@tool("check_lm_transaction", description=T.check_lm_transaction.description)
def check_validated(date: str, runtime: ToolRuntime[Context], montant: Optional[int] = None) -> str:
    messages_client = [m.content for m in runtime.state["messages"] if isinstance(m, HumanMessage)]
    date_valide = True   # TODO 6 : remplacez True par la vérification avec date_given
    if not date_valide:
        return json.dumps({"erreur": f"La date {date} n'a pas été donnée par le client.",
                           "indication": "Demande au client la date de son remboursement, puis rappelle ce tool."},
                          ensure_ascii=False)
    return json.dumps(T.fetch_lm_transactions(runtime.context.client_id, date, montant), ensure_ascii=False)


TRACE_DATE2, RESULT_DATE2, _ = run(build_agent(check=check_validated), NODATE)
TRACE_DATE2.table()

In [ ]:
print(final_answer(RESULT_DATE2))
TRACE_KO, RESULT_KO, _ = run(build_agent(check=check_validated), KOUASSI)
LECTURES_KO = [s["result"] for s in TRACE_KO.tool_calls() if s["name"] == "check_lm_transaction"]
print("\nCas avec « hier » : la validation laisse passer :", any('"transaction_id"' in r for r in LECTURES_KO))

## <a id="evaluer-la-trajectoire-et-lissue"></a>Évaluer la trajectoire et l'issue

Un test passe si l'agent a appelé les bons tools et atteint la bonne issue, pas seulement si la réponse est polie. Cinq scénarios, deux agents :

- **avant** : descriptions interverties, aucune validation, aucune limite d'appels ;
- **après** : les trois parades.

Pour le scénario 5, l'API de paiement n'aboutit jamais (`check_pending`).

In [ ]:
SANS_DATE_NI_HIER = "J'ai remboursé mon échéance hier mais mon dossier indique un impayé."


def count(table):
    return int(sql(f"SELECT COUNT(*) AS n FROM {table}").iloc[0, 0])


def check_result_found(trace):
    for s in trace.tool_calls():
        if s["name"] == "check_lm_transaction" and s["result"].startswith("{") and json.loads(s["result"]).get("transactions"):
            return True
    return False


SCENARIOS = {
    "1 réussie, non rattachée": ("CLI-0002", KOUASSI, False,
                                 lambda tr, res, err: err is None and {"check_lm_transaction", "propose_regularisation"} <= {s["name"] for s in tr.tool_calls()}),
    "2 transaction en échec": ("CLI-0003", SANS_DATE_NI_HIER, False,
                               lambda tr, res, err: err is None and "check_lm_transaction" in {s["name"] for s in tr.tool_calls()}
                               and "propose_regularisation" not in {s["name"] for s in tr.tool_calls()}),
    "3 transaction introuvable": ("CLI-0004", SANS_DATE_NI_HIER, False,
                                  lambda tr, res, err: err is None and "create_claim" in {s["name"] for s in tr.tool_calls()}),
    "4 date non donnée": ("CLI-0002", NODATE, False,
                          lambda tr, res, err: err is None and not check_result_found(tr) and "?" in final_answer(res)),
    "5 API qui n'aboutit pas": ("CLI-0002", KOUASSI, True,
                                lambda tr, res, err: err is None and [s["name"] for s in tr.tool_calls()].count("check_lm_transaction") <= 4),
}


def evaluate(label, descriptions, check_ok, limiter):
    lignes = []
    for nom, (client, message, pending, verdict) in SCENARIOS.items():
        agent = build_agent(check=check_pending if pending else check_ok, descriptions=descriptions, limiter=limiter)
        for _ in range(RUNS):
            tracer, result, err = run(agent, message, client, limit=30)
            lignes.append({"scénario": nom, "agent": label, "ok": bool(verdict(tracer, result, err))})
    return pd.DataFrame(lignes)


EVALUATION = pd.concat([evaluate("avant", INTERVERTIES, None, None),
                        evaluate("après", CORRIGEES, check_validated, LIMITER)])
TABLE = EVALUATION.pivot_table(index="scénario", columns="agent", values="ok", aggfunc="mean")[["avant", "après"]].round(2)
TABLE

## <a id="sur-le-chat"></a>Sur le chat

L'agent corrigé sur l'interface Gradio : chaque tool appelé s'affiche sous forme de bloc.

In [ ]:
import chat
import front

demo = chat.create(build_agent(check=check_validated, descriptions=CORRIGEES, limiter=LIMITER))
front.launch(demo)

In [ ]:
demo.close()
clear_langfuse()

## Contrôle

Cette cellule affiche **ÉCHEC** tant que les `# TODO` ne sont pas faits.

In [ ]:
try:
    noms_ok = [s["name"] for s in TRACE_OK.tool_calls()]
    assert TOOLS_DEMANDES and sorted(n for n, _ in TOOLS_DEMANDES) == sorted(noms_ok), "tools demandés, lus dans RESULT_OK[\"messages\"] (TODO 1)"
    assert ERREUR_LOOP == "GraphRecursionError" or APPELS_BOUCLE and APPELS_BOUCLE >= 5, "la boucle n'a pas été reproduite : relancez la cellule de l'erreur 1"
    noms = [s["name"] for s in TRACE_LOOP.tool_calls()]
    assert TOOL_BOUCLE == max(set(noms), key=noms.count) and APPELS_BOUCLE == noms.count(TOOL_BOUCLE), "tool et nombre d'appels de la boucle (TODO 2)"
    assert isinstance(LIMITER, ToolCallLimitMiddleware) and LIMITER.tool_name == "check_lm_transaction" and LIMITER.exit_behavior == "end", "LIMITER : ToolCallLimitMiddleware sur check_lm_transaction avec exit_behavior=\"end\" (TODO 3)"
    assert LIMITER.run_limit and 1 <= LIMITER.run_limit <= 5, "run_limit entre 1 et 5 (TODO 3)"
    assert APPELS_LIMIT <= LIMITE_APPELS and ERREUR_LIMIT is None and RESULT_LIMIT, "avec la limite, l'agent doit conclure sans dépasser run_limit (TODO 3)"
    assert PREMIERE_DECISION == TRACE_BAD.decisions()[0] and BON_CHOIX == (PREMIERE_DECISION == ["check_lm_transaction"]), "première décision du modèle (TODO 4)"
    assert APRES_2[0] / APRES_2[1] >= 0.75, "descriptions corrigées : check_lm_transaction doit être le premier tool presque toujours (TODO 5)"
    assert ORIGINE_DATE in ("tool", "inventée"), "l'erreur 3 n'a pas été reproduite : relancez la cellule de l'erreur 3"
    assert not check_result_found(TRACE_DATE2), "la validation doit empêcher la lecture sur une date non donnée (TODO 6)"
    assert any('"transaction_id"' in r for r in LECTURES_KO), "la validation ne doit pas refuser la date « hier » (TODO 6)"
    assert TABLE["après"].mean() >= 0.8 and TABLE["après"].mean() > TABLE["avant"].mean(), "l'agent corrigé doit réussir au moins 80 % des scénarios, mieux que l'agent initial"
    print("OK")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else 'terminez les # TODO et relancez les cellules.'}")